In [ ]:
import pandas as pd

df_telemetry_log = pd.read_csv('telemetry_log_20260622-201715.csv', skiprows=[0])
df_telemetry_log

: 

In [ ]:
df_distance_log = pd.read_csv('distance_log_20260622-201715.csv')
df_distance_log

## Análise da Taxa de Pacotes

In [ ]:
indice_tempo = pd.to_timedelta(df_telemetry_log['timestamp'], unit='ms')

df_telemetry_log['pacotes_por_segundo'] = (
    df_telemetry_log
    .set_index(indice_tempo)['timestamp']
    .rolling('1s')
    .count()
    .values
)

df_telemetry_log

In [ ]:
# Casos em que o pacote corrompeu
df_telemetry_log[df_telemetry_log['Is_Valid_Mavlink'] == 0]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

tempo_em_segundos = df_telemetry_log['timestamp'] / 1000

plt.plot(tempo_em_segundos, df_telemetry_log['pacotes_por_segundo'], 
         color='#1f77b4', linewidth=1.5)

plt.title('Frequência de Pacotes por Segundo ao Longo do Tempo', fontsize=14, pad=15)
plt.xlabel('Tempo decorrido (segundos)', fontsize=12)
plt.ylabel('Frequência (Pacotes / Segundo)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# timestamp em segundos para o eixo X
df_telemetry_log['timestamp_seconds'] = df_telemetry_log['timestamp'] / 1000
df_distance_log['timestamp_seconds'] = df_distance_log['timestamp'] / 1000

# 3. Plotar o gráfico principal
plt.plot(df_telemetry_log['timestamp_seconds'], df_telemetry_log['RSSI'], label='Sinal RSSI', color='dodgerblue', alpha=0.8)

# 4. Adicionar os indicadores discretos de distância
y_text_position = df_telemetry_log['RSSI'].max() + 2 

for index, row in df_distance_log.iterrows():
    ts = row['timestamp_seconds']
    dist = row['distance_origin_meters']
    
    # Desenhar a linha vertical pontilhada
    plt.axvline(x=ts, color='crimson', linestyle='--', alpha=0.7)
    
    # Textos orientados na horizontal (rotation=0) e centralizados com a linha
    plt.text(ts, y_text_position, f'{dist}m', color='crimson', 
             rotation=0, verticalalignment='bottom', horizontalalignment='center',
             fontweight='bold') # Adicionei negrito para destacar mais

# 5. Configurações estéticas
plt.xlabel('Timestamp (s)')
plt.ylabel('RSSI (dBm)')
#plt.title('Variação do RSSI ao Longo do Tempo e Distâncias Correspondentes', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')

# Aumentar o limite superior do eixo Y para o texto não ficar cortado pela borda do gráfico
plt.ylim(df_telemetry_log['RSSI'].min() - 5, df_telemetry_log['RSSI'].max() + 8)

plt.tight_layout()
plt.savefig('grafico_rssi_distancia_horizontal.png', dpi=150)

In [ ]:
plt.figure(figsize=(12, 6))

tempo_em_segundos = df_telemetry_log['timestamp'] / 1000

plt.plot(tempo_em_segundos, df_telemetry_log['SNR'], 
         color='#1f77b4', linewidth=1.5)

plt.title('SNR ao Longo do Tempo', fontsize=14, pad=15)
plt.xlabel('Tempo decorrido (segundos)', fontsize=12)
plt.ylabel('SNR', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

tempo_em_segundos = df_telemetry_log['timestamp'] / 1000

plt.plot(tempo_em_segundos, df_telemetry_log['Is_Valid_Mavlink'], 
         color='#1f77b4', linewidth=1.5)

plt.title('Integridade ao Longo do Tempo', fontsize=14, pad=15)
plt.xlabel('Tempo decorrido (segundos)', fontsize=12)
plt.ylabel('Integridade', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
# Casos de falha / total de pacotes
quantidade_falhas = df_telemetry_log[df_telemetry_log['Is_Valid_Mavlink'] == 0].shape[0]
total_pacotes = df_telemetry_log.shape[0]
proporcao_falhas = quantidade_falhas / total_pacotes if total_pacotes > 0 else 0

print(proporcao_falhas)